In [ ]:
!uname -a && cat /etc/*release

In [28]:
!pwd
!ls -la

/content
total 20
drwxr-xr-x 1 root root 4096 Jun 23 14:53 .
drwxr-xr-x 1 root root 4096 Jun 23 13:47 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 2 root root 4096 Jun 23 14:53 .ipynb_checkpoints
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [1]:
!nvidia-smi

Tue Jun 23 08:07:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
device_query_str = '''
#include <stdio.h>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);

    if (deviceCount == 0) {
        printf("Nessun dispositivo CUDA trovato.\\n");
        return 1;
    }

    for (int dev = 0; dev < deviceCount; ++dev) {
        cudaDeviceProp deviceProp;
        cudaGetDeviceProperties(&deviceProp, dev);

        printf("\\n--- DEVICE %d: \\"%s\\" ---\\n", dev, deviceProp.name);
        printf("  CUDA Capability Major/Minor version:          %d.%d\\n", deviceProp.major, deviceProp.minor);
        printf("  Total amount of global memory:                %.2f GBytes\\n", (float)deviceProp.totalGlobalMem / (1024 * 1024 * 1024));
        printf("  Number of multiprocessors (SMs):              %d\\n", deviceProp.multiProcessorCount);
        printf("  Total amount of constant memory:              %lu bytes\\n", deviceProp.totalConstMem);
        printf("  Total amount of shared memory per block:      %lu bytes\\n", deviceProp.sharedMemPerBlock);
        printf("  Total number of registers available per block: %d\\n", deviceProp.regsPerBlock);
        printf("  Warp size:                                    %d\\n", deviceProp.warpSize);
        printf("  Maximum threads per multiprocessor (SM):      %d\\n", deviceProp.maxThreadsPerMultiProcessor);
        printf("  Maximum threads per block:                    %d\\n", deviceProp.maxThreadsPerBlock);
        printf("  Max dimension size of a thread block (x,y,z): %d, %d, %d\\n", deviceProp.maxThreadsDim[0], deviceProp.maxThreadsDim[1], deviceProp.maxThreadsDim[2]);
        printf("  Max dimension size of a grid size    (x,y,z): %d, %d, %d\\n", deviceProp.maxGridSize[0], deviceProp.maxGridSize[1], deviceProp.maxGridSize[2]);
    }
    return 0;
}
'''
with open('deviceQuery.cu', 'w') as f:
    f.write(device_query_str)

In [19]:
!nvcc -arch=sm_75 deviceQuery.cu -o test

In [10]:
!/content/test


--- DEVICE 0: "Tesla T4" ---
  CUDA Capability Major/Minor version:          7.5
  Total amount of global memory:                14.56 GBytes
  Number of multiprocessors (SMs):              40
  Total amount of constant memory:              65536 bytes
  Total amount of shared memory per block:      49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                    32
  Maximum threads per multiprocessor (SM):      1024
  Maximum threads per block:                    1024
  Max dimension size of a thread block (x,y,z): 1024, 1024, 64
  Max dimension size of a grid size    (x,y,z): 2147483647, 65535, 65535


In [11]:
!nvprof /content/test

==5746== NVPROF is profiling process 5746, command: /content/test

--- DEVICE 0: "Tesla T4" ---
  CUDA Capability Major/Minor version:          7.5
  Total amount of global memory:                14.56 GBytes
  Number of multiprocessors (SMs):              40
  Total amount of constant memory:              65536 bytes
  Total amount of shared memory per block:      49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                    32
  Maximum threads per multiprocessor (SM):      1024
  Maximum threads per block:                    1024
  Max dimension size of a thread block (x,y,z): 1024, 1024, 64
  Max dimension size of a grid size    (x,y,z): 2147483647, 65535, 65535
==5746== Profiling application: /content/test
==5746== Profiling result:
No kernels were profiled.
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
      API calls:   51.87%  2.3356ms       114  20.488us      89ns  1.1759ms  cuDeviceGetAtt

In [16]:
transfer_str = '''
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    const int num_elements = 100 * 1024 * 1024;
    const size_t size = num_elements * sizeof(float);

    float *h_data = (float *)malloc(size);
    float *d_data;

    // Allocazione sulla memoria globale della GPU
    cudaMalloc(&d_data, size);

    // Il collo di bottiglia: Trasferimento Host-to-Device
    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);

    cudaFree(d_data);
    free(h_data);
    return 0;
}
'''
with open('transfer.cu', 'w') as f:
    f.write(transfer_str)

In [20]:
!nvcc -arch=sm_75 transfer.cu -o transfer

In [21]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys profile -t cuda,nvtx,osrt -o report_host_device --stats=true --force-overwrite=true ./transfer

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
Generating '/tmp/nsys-report-bfd6.qdstrm'
[1/8] [========================100%] report_host_device.nsys-rep
[2/8] [========================100%] report_host_device.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/report_host_device.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  ----------------------
     76.3      602,496,804         15  40,166,453.6  14,080,955.0     1,391  301,452,989  78,124,842.7  poll                  
     23.2      182,990,959        513     356,707.5      14,992.0     1,017   20,689,570   1,339,068.2  ioctl              

In [15]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.29
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [36]:
%%writefile matrix_seq.c
#define n 5000

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

int main(int argc,char **argv) {
  int i, j, k;

  double ( *a )[n] = malloc(sizeof(double[n][n]));
  double ( *b )[n] = malloc(sizeof(double[n][n]));
  double ( *c )[n] = malloc(sizeof(double[n][n]));

  for (i=0; i<n; i++)
     for (j=0; j<n; j++) {
        a[i][j] = 2.0;
        b[i][j] = 3.0;
        c[i][j] = 0.0;
     }

  double start_time = omp_get_wtime();

  // L'hotspot computazionale (Triplo ciclo for)
  // ATTENZIONE: Nessun #pragma omp qui, deve rimanere sequenziale!
  for (i=0; i<n; ++i)
     for (k=0; k<n; k++)
        for (j=0; j<n; ++j)
           c[i][j] += a[i][k]*b[k][j];


  double end_time = omp_get_wtime();
  double elapsed_tempo_hotspot = end_time - start_time;

  printf("Dimensione Matrice N: %d\n", n);
  printf("Tempo di esecuzione Hotspot: %f secondi\n", elapsed_tempo_hotspot);

  FILE *f = fopen("mat-res.txt", "w");
  if (!f) {
     perror("fopen");
      return 1;
  }

  fprintf(f, "%d\n\n", n);
  for (int i = 0; i < 1000; i++) {
     for (int j = 0; j < 1000; j++) {
        fprintf(f, "%.0f ", c[i][j]);
     }
     fprintf(f, "\n");
  }

  fclose(f);
  free(a);
  free(b);
  free(c);
  return 0;
}

Overwriting matrix_seq.c


In [38]:
# Compilazione (aggiunto -fopenmp per abilitare la libreria del tempo)
!gcc -O3 -fopenmp matrix_seq.c -o matrix_seq



In [ ]:
# Esecuzione
!time ./matrix_seq 2000